# 02 - Customer Analysis

**E-Commerce Revenue & Customer Analytics**

This notebook digs into customer-level behavior:
- Repeat purchase rate and order frequency
- Customer revenue distribution & top customers
- New vs. returning customer trends
- Customer Lifetime Value (CLV)
- Geographic customer distribution

Run `python scripts/run_pipeline.py` from the project root before executing this notebook.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROCESSED_DIR = os.path.join("..", "data", "processed")

In [ ]:
customer_features = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features.csv"), parse_dates=["signup_date", "first_order_date", "last_order_date"])
orders = pd.read_csv(os.path.join(PROCESSED_DIR, "orders_enriched.csv"), parse_dates=["order_date"])
clv = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_lifetime_value.csv"))

print(f"customer_features: {customer_features.shape}")
print(f"orders: {orders.shape}")
print(f"clv: {clv.shape}")

## 1. Headline customer metrics

In [ ]:
active_customers = customer_features[customer_features["order_count"] > 0]

total_customers = len(customer_features)
active_count = len(active_customers)
repeat_count = active_customers["is_repeat_customer"].sum()
repeat_rate = repeat_count / active_count * 100

print(f"Total registered customers: {total_customers:,}")
print(f"Active customers (>=1 order): {active_count:,} ({active_count/total_customers*100:.1f}% of registered)")
print(f"Repeat customers (>=2 orders): {repeat_count:,}")
print(f"Repeat purchase rate: {repeat_rate:.2f}%")
print(f"Average orders per active customer: {active_customers['order_count'].mean():.2f}")
print(f"Average revenue per active customer: Rs {active_customers['total_revenue'].mean():,.2f}")

**What does this tell the business?** A repeat purchase rate in the mid-30s% is a healthy baseline for a broad-catalog e-commerce business, but there's clear room to grow -- roughly two-thirds of active customers have only ordered once. Converting even a modest slice of one-time buyers into repeat buyers has an outsized effect on revenue, since repeat customers spend significantly more per capita (see below).

In [ ]:
order_dist = active_customers["order_count"].value_counts().sort_index().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(order_dist.index.astype(str), order_dist.values, color="#4C72B0")
ax.set_title("Distribution of Orders per Customer (top 10 order counts)", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Orders")
ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.show()

## 2. Customer revenue distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(active_customers["total_revenue"], bins=50, ax=ax, color="#55A868")
ax.set_title("Customer Lifetime Revenue Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Total Revenue per Customer (Rs)")
ax.set_ylabel("Number of Customers")
ax.set_xlim(0, active_customers["total_revenue"].quantile(0.98))  # trim extreme tail for readability
plt.tight_layout()
plt.show()

**What does this tell the business?** Like most e-commerce businesses, customer revenue is right-skewed: a large mass of low-spend customers and a long tail of high-value customers. This is exactly what RFM/CLV segmentation (next section and notebook 03) is designed to act on -- treating all customers the same wastes marketing budget on low-value segments.

In [ ]:
revenue_by_repeat = active_customers.groupby("is_repeat_customer")["total_revenue"].agg(["mean", "median", "count"])
revenue_by_repeat.index = ["One-time buyers", "Repeat buyers"]
revenue_by_repeat.columns = ["avg_revenue", "median_revenue", "customer_count"]
revenue_by_repeat

## 3. Top 15 customers by lifetime revenue

In [ ]:
top_customers = active_customers.sort_values("total_revenue", ascending=False).head(15)[
    ["customer_id", "customer_name", "city", "state", "order_count", "total_revenue", "average_order_value"]
]
top_customers

## 4. New vs. returning customer trend

In [ ]:
valid_orders = orders[orders["order_status"] != "Cancelled"].copy()
valid_orders["is_first_order"] = valid_orders["is_first_order"].astype(bool)

monthly_split = valid_orders.groupby(["year_month", "is_first_order"])["revenue"].sum().unstack(fill_value=0)
monthly_split.columns = ["Returning Customer Revenue", "New Customer Revenue"]
monthly_split = monthly_split.sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(monthly_split.index, monthly_split["New Customer Revenue"], monthly_split["Returning Customer Revenue"],
             labels=["New Customer Revenue", "Returning Customer Revenue"],
             colors=["#DD8452", "#4C72B0"])
ax.set_title("Monthly Revenue: New vs. Returning Customers", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (Rs)")
ax.legend(loc="upper left")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

**What does this tell the business?** If the returning-customer share of revenue grows over time, retention efforts are compounding as the customer base matures -- a good sign for long-term unit economics. If new-customer revenue dominates every month with little growth in the returning share, the business is acquisition-dependent, which is typically a more expensive and less durable growth model.

## 5. Customer Lifetime Value (CLV)

See `scripts/generate_dashboard_data.py::compute_clv()` for the full formula and assumptions. In short:

```
CLV = Average Order Value x (Orders per Month of Tenure) x (Estimated Lifespan in Months, capped at 36)
```

This is a **practical, assumption-based estimate** meant to rank and segment customers by long-term value potential -- it is not a churn-model-based prediction.

In [ ]:
print(f"Average CLV: Rs {clv['clv'].mean():,.2f}")
print(f"Median CLV: Rs {clv['clv'].median():,.2f}")
print()
clv_by_tier = clv.groupby("clv_tier", observed=True).agg(
    customers=("customer_id", "count"),
    avg_clv=("clv", "mean"),
    total_clv=("clv", "sum"),
).round(2)
clv_by_tier

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(clv["clv"].clip(upper=clv["clv"].quantile(0.95)), bins=50, ax=ax, color="#8172B2")
ax.set_title("Customer Lifetime Value Distribution (95th percentile capped for readability)", fontsize=13, fontweight="bold")
ax.set_xlabel("Estimated CLV (Rs)")
ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.show()

**What does this tell the business?** The 'Top' CLV quartile represents the customers worth the most retention investment (loyalty perks, personalized outreach, early access to sales). The 'Low' quartile is where acquisition cost needs the most scrutiny -- if the cost to acquire those customers exceeds their estimated lifetime value, that acquisition channel is losing money.

## 6. Geographic customer distribution

In [ ]:
geo_customers = active_customers.groupby("state").agg(
    customers=("customer_id", "count"),
    avg_revenue=("total_revenue", "mean"),
).reset_index().sort_values("customers", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(geo_customers["state"], geo_customers["customers"], color="#64B5CD")
ax.set_title("Top 15 States by Active Customer Count", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Active Customers")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

Customer behavior in this dataset shows classic e-commerce patterns: a right-skewed revenue distribution, a repeat purchase rate with clear room to improve, and a small set of high-CLV customers driving disproportionate value. The next notebook (`03_rfm_segmentation.ipynb`) formalizes this into actionable RFM segments.